In [ ]:
# -*- coding: utf-8 -*-
"""
DASH APP — multi-video particle-track viewer and classifier.

KEY FEATURES:
- Use t >= t_min and a minimum track-length cut for plotting and analysis.
- Estimate terminal velocity using a quadratic late-time fit, with linear fallback.
- Separate clusters from single particles using the luminosity-based N_est threshold.
- Classify single particles as free or trapped using the fitted 2D GMM.
- Show trajectory plots, velocity-space plots, histograms, and |Vx| versus laser power.
- Allow interactive track selection and deletion.

"""

import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import re
from dataclasses import dataclass

import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
from sklearn.mixture import GaussianMixture

import warnings
warnings.filterwarnings(
    "ignore",
    message="KMeans is known to have a memory leak on Windows with MKL"
)

import dash
from dash import Dash, dcc, html, Input, Output, State, no_update
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# ============================================================
# ---------------- USER SETTINGS (ONLY EDIT HERE) -------------
# ============================================================

BASE_DIR = r"D:\Manchester\Manchester\year four\lab\Input Videos\videos 2"

# physics / calibration
FPS = 990.0
CM_PER_PIXEL = 0.23 / 200.0   # = 0.00115 cm/px

# t-selector
T_MIN_DEFAULT = 0.06
HIDE_TRACKS_WITHOUT_POST_TMIN = True

# track-length selector (after t_min cut)
MIN_POINTS_DEFAULT = 5

# velocity fitting
VEL_MIN_POINTS = 3
FIT_AFTER_Y_PEAK = True

# plot
HIST_BINS = 14
HITBOX_WIDTH = 18
N_EST_HIST_BINS = 26
N_EST_XMAX = 3.0

# heatmap grid
HEATMAP_NX = 140
HEATMAP_NY = 140

# colors
COL_FREE = "red"
COL_TRAPPED = "green"
COL_CLUSTER = "orange"

ALPHA_FREE = 1.0
ALPHA_TRAPPED = 0.85
ALPHA_CLUSTER = 0.95

# server
HOST = "127.0.0.1"
PORT = 8061
DEBUG = False

# ONLY load this pattern
ONLY_SUFFIX = "__manually_added__linkfmt__subpixel__xfm.csv"

# special combined key
ALL_KEY = "ALL_COMBINED (all params)"

LASER_POWER_MAP_MW = {
    350.0: 641.0,
    355.0: 317.0,
    360.0: 133.0,
    365.0: 46.2,
}

# size / luminosity model
SIZE_M = 2.0 / 3.0
CLUSTER_N_THRESHOLD_DEFAULT = 1.4
FREE_S_THRESHOLD_DEFAULT = 0.95
INCLUDE_CLUSTERS_IN_GMM_DEFAULT = True  
LUM_PEAK_HIST_BINS = 40
R1_UM = 2.9

# effective gravity used for apparent relaxation-time estimate
G_PRIME_M_S2 = 7.91
G_PRIME_CM_S2 = G_PRIME_M_S2 * 100.0

# ============================================================
# ------------------------ utilities --------------------------
# ============================================================

def extract_laser_us_from_key(vkey):
    m = re.search(r'(\d+(?:\.\d+)?)us', str(vkey), flags=re.IGNORECASE)
    if m:
        return float(m.group(1))
    return np.nan

def extract_temp_k_from_key(vkey):
    s = str(vkey)

    m_mk = re.search(r'(\d+(?:\.\d+)?)\s*mk', s, flags=re.IGNORECASE)
    if m_mk:
        return float(m_mk.group(1)) / 1000.0

    m_k = re.search(r'(\d+(?:\.\d+)?)\s*k', s, flags=re.IGNORECASE)
    if m_k:
        return float(m_k.group(1))

    return np.nan

def canonical_temp_k_for_grouping(temp_k):
    if not np.isfinite(temp_k):
        return np.nan

    temp_k = float(temp_k)

    if abs(temp_k - 0.610) < 0.015:
        return 0.600

    return temp_k
    
def laser_us_to_power_mw(us):
    if not np.isfinite(us):
        return np.nan
    return float(LASER_POWER_MAP_MW.get(float(us), np.nan))
    
def build_deleted_output_path(csv_path):
    folder = os.path.dirname(csv_path)
    base, ext = os.path.splitext(os.path.basename(csv_path))
    return os.path.join(folder, f"{base}__deleted_tracks{ext}")
    
def _norm(s):
    return str(s).strip().lower().replace(" ", "").replace("_", "").replace("-", "")

def _auto_range(vals, pad_frac=0.03, fallback=(0.0, 1.0)):
    vals = np.asarray(vals, float)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return [float(fallback[0]), float(fallback[1])]
    vmin = float(np.min(vals))
    vmax = float(np.max(vals))
    if vmax == vmin:
        return [vmin - 1.0, vmax + 1.0]
    pad = (vmax - vmin) * float(pad_frac)
    return [vmin - pad, vmax + pad]

def _format_line_eqn(a, b, x_name="x", y_name="y"):
    a = float(a)
    b = float(b)
    sign = "+" if b >= 0 else "-"
    return f"{y_name} = {a:.4g}{x_name} {sign} {abs(b):.4g}"
    
def _pick_existing_col(df, candidates):
    cols = list(df.columns)

    for c in candidates:
        if c in df.columns:
            return c

    m = {_norm(c): c for c in cols}
    for c in candidates:
        cn = _norm(c)
        if cn in m:
            return m[cn]

    for c in candidates:
        cn = _norm(c)
        for cc in cols:
            if cn and (cn in _norm(cc)):
                return cc
    return None

def _trim_by_t(t, x, y, t_min):
    t = np.asarray(t, float)
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    m = np.isfinite(t) & np.isfinite(x) & np.isfinite(y) & (t >= float(t_min))
    if not np.any(m):
        return np.asarray([], float), np.asarray([], float), np.asarray([], float)
    return t[m], x[m], y[m]

def _rmsd_from_fit(y, yfit):
    r = y - yfit
    msd = np.nanmean(r * r)
    return float(np.sqrt(msd)) if np.isfinite(msd) else np.nan

def find_lum_peak_from_values(lum_vals, n_bins=LUM_PEAK_HIST_BINS):
    """
    Find single-particle luminosity peak Lum_1 from the histogram
    of per-track mean luminosities.
    """
    lum = np.asarray(lum_vals, float)
    lum = lum[np.isfinite(lum) & (lum > 0)]
    if lum.size == 0:
        return np.nan

    vmin = float(np.min(lum))
    vmax = float(np.max(lum))
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
        return np.nan

    padding = 0.05 * (vmax - vmin)
    edges = np.linspace(vmin - padding, vmax + padding, int(n_bins))
    if edges.size < 2:
        return np.nan

    counts, edges = np.histogram(lum, bins=edges)
    if counts.size == 0 or np.sum(counts) <= 0:
        return np.nan

    centres = 0.5 * (edges[:-1] + edges[1:])
    i_peak = int(np.argmax(counts))
    return float(centres[i_peak])



def _make_code1_style_track_id_mapping(df, uid_col):
    """
    track_id = 1..N in the order unique_id first appears in the file.
    """
    uid_order = pd.unique(df[uid_col].astype(str))
    mp = {u: i + 1 for i, u in enumerate(uid_order)}
    return mp



def compute_track_level_metrics_from_tracks(df_tracks, cm_per_pixel, fps):
    uid_col = _pick_existing_col(df_tracks, ["unique_id", "Unique_ID"])
    frame_col = _pick_existing_col(df_tracks, ["frame", "Frame"])
    x_col = _pick_existing_col(df_tracks, ["x", "X"])
    y_col = _pick_existing_col(df_tracks, ["y", "Y"])
    lum_col = _pick_existing_col(
        df_tracks,
        ["raw_luminosity", "raw luminosity", "raw_luminocity", "raw_luminosity ",
         "mean_luminosity", "mean luminosity", "Mean_Lum", "mean_lum", "luminosity", "lum"]
    )

    need = [uid_col, frame_col, x_col, y_col]
    if any(c is None for c in need):
        raise ValueError(f"[Tracks] Missing required columns. Have: {list(df_tracks.columns)}")

    df = df_tracks.copy()
    df = df.dropna(subset=[uid_col, frame_col, x_col, y_col]).copy()

    df[frame_col] = pd.to_numeric(df[frame_col], errors="coerce")
    df[x_col] = pd.to_numeric(df[x_col], errors="coerce")
    df[y_col] = pd.to_numeric(df[y_col], errors="coerce")
    df = df.dropna(subset=[frame_col, x_col, y_col]).copy()
    df[frame_col] = df[frame_col].astype(int)

    mp = _make_code1_style_track_id_mapping(df, uid_col)
    df["_track_id"] = df[uid_col].astype(str).map(mp).astype(int)

    df["_x_cm"] = df[x_col].to_numpy(dtype=float) * float(cm_per_pixel)
    df["_y_cm"] = df[y_col].to_numpy(dtype=float) * float(cm_per_pixel)

    mean_lum_by_tid = {}
    if lum_col is not None and lum_col in df.columns:
        df["_lum_tmp"] = pd.to_numeric(df[lum_col], errors="coerce")
        for tid, g in df.groupby("_track_id", sort=True):
            lumv = g["_lum_tmp"].to_numpy(dtype=float)
            lumv = lumv[np.isfinite(lumv) & (lumv > 0)]
            mean_lum_by_tid[int(tid)] = float(np.mean(lumv)) if lumv.size > 0 else np.nan
    else:
        for tid in df["_track_id"].unique():
            mean_lum_by_tid[int(tid)] = np.nan

    lum1 = find_lum_peak_from_values(list(mean_lum_by_tid.values()), n_bins=LUM_PEAK_HIST_BINS)

    global_fmin = int(df[frame_col].min())
    track_cache = {}
    t_end_max = 0.0

    for tid, g in df.groupby("_track_id", sort=True):
        g = g.sort_values(frame_col)
        frames = g[frame_col].to_numpy(dtype=int)
        x = g["_x_cm"].to_numpy(dtype=float)
        y = g["_y_cm"].to_numpy(dtype=float)
        t = (frames - global_fmin) / float(fps)

        if len(t) < 2:
            continue

        uid0 = g[uid_col].iloc[0] if uid_col in g.columns else ""


        dX = float(x[-1] - x[0])
        dY = float(y[-1] - y[0])
        tort = 0.0
        for i in range(len(x) - 1):
            if np.isfinite(x[i]) and np.isfinite(y[i]) and np.isfinite(x[i+1]) and np.isfinite(y[i+1]):
                tort += float(np.hypot(x[i+1] - x[i], y[i+1] - y[i]))
            else:
                tort = np.nan
                break
        net_jump = float(np.hypot(dX, dY))
        s_full = float(net_jump / tort) if (np.isfinite(tort) and tort > 0) else np.nan


        mfit = np.isfinite(t) & np.isfinite(x) & np.isfinite(y)
        if mfit.sum() >= 2:
            tt = t[mfit]
            xx = x[mfit]
            yy = y[mfit]
            p1x = np.polyfit(tt, xx, 1)
            p1y = np.polyfit(tt, yy, 1)
            xfit = np.polyval(p1x, tt)
            yfit = np.polyval(p1y, tt)
            rmsd = float(np.sqrt((_rmsd_from_fit(xx, xfit) ** 2) + (_rmsd_from_fit(yy, yfit) ** 2)))
            vx_lin = float(p1x[0])
            vy_lin = float(p1y[0])
            speed_lin = float(np.hypot(vx_lin, vy_lin))
            dt = float(tt[-1] - tt[0])
            denom = speed_lin * dt if (np.isfinite(speed_lin) and np.isfinite(dt)) else np.nan
            w_full = float(rmsd / denom) if (np.isfinite(denom) and denom != 0.0) else np.nan
        else:
            w_full = np.nan

        lum = float(mean_lum_by_tid.get(int(tid), np.nan))

        if np.isfinite(lum) and np.isfinite(lum1) and lum > 0 and lum1 > 0:
            n_est = float((lum / lum1) ** (1.0 / SIZE_M))
            r_est_um = float(R1_UM * (lum / lum1) ** (1.0 / (3.0 * SIZE_M)))
            is_cluster = bool(n_est >= CLUSTER_N_THRESHOLD_DEFAULT)
        else:
            n_est = np.nan
            r_est_um = np.nan
            is_cluster = False

        track_cache[int(tid)] = dict(
            tid=int(tid),
            unique_id=str(uid0),
            t=t, x=x, y=y,
            s_full=s_full,
            w=w_full,
            lum=lum,
            lum1=lum1,
            n_est=n_est,
            r_est_um=r_est_um,
            is_cluster=is_cluster,
        )
        t_end_max = max(t_end_max, float(np.nanmax(t)))

    return track_cache, float(t_end_max), float(lum1) if np.isfinite(lum1) else np.nan


def _fit_quad_terminal_velocity(t, z):
    t = np.asarray(t, float)
    z = np.asarray(z, float)
    m = np.isfinite(t) & np.isfinite(z)
    t = t[m]; z = z[m]
    if t.size < 3:
        raise ValueError("need >=3 points for quadratic")
    if np.unique(t).size < 3:
        raise ValueError("need >=3 unique t")
    a, b, _c = np.polyfit(t, z, 2)
    t_end = float(np.nanmax(t))
    v_end = float(2.0 * a * t_end + b)
    return v_end


def terminal_v_quadratic_with_fallback(track_dict, t_min):
    t = np.asarray(track_dict["t"], float)
    x = np.asarray(track_dict["x"], float)
    y = np.asarray(track_dict["y"], float)

    frame_all = np.rint(t * float(FPS)).astype(int)


    t2, x2, y2, f2 = _trim_by_t_with_frames(t, x, y, frame_all, float(t_min))


    t2, x2, y2, f2 = _split_until_first_gap(t2, x2, y2, f2)

    if t2.size < VEL_MIN_POINTS:
        return None

    t_fit, x_fit, y_fit = t2, x2, y2

    if FIT_AFTER_Y_PEAK:
        try:
            i_peak = int(np.nanargmax(y2))
            i0 = min(i_peak + 1, len(y2))

            if (len(y2) - i0) >= VEL_MIN_POINTS:
                t_fit = t2[i0:]
                x_fit = x2[i0:]
                y_fit = y2[i0:]
        except Exception:
            pass

    try:
        vx = _fit_quad_terminal_velocity(t_fit, x_fit)
        vy = _fit_quad_terminal_velocity(t_fit, y_fit)
    except Exception:
        try:
            vx = float(np.polyfit(t_fit, x_fit, 1)[0])
            vy = float(np.polyfit(t_fit, y_fit, 1)[0])
        except Exception:
            return None

    return dict(vx=float(vx), vy=float(vy), speed=float(np.hypot(vx, vy)))


# ============================================================
# ---------------- 1D two-Gaussian fit (COUNTS) ---------------
# ============================================================

def _hist_edges_from_values(values, bins, fixed_range=None):
    values = np.asarray(values, float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.linspace(-1.0, 1.0, int(bins) + 1)
    if fixed_range is not None:
        lo, hi = float(fixed_range[0]), float(fixed_range[1])
    else:
        lo = float(np.min(values))
        hi = float(np.max(values))
        if hi == lo:
            hi = lo + 1.0
    return np.linspace(lo, hi, int(bins) + 1)

def _hist_counts(values, edges):
    v = np.asarray(values, float)
    v = v[np.isfinite(v)]
    hist, _ = np.histogram(v, bins=edges)
    centers = 0.5 * (edges[:-1] + edges[1:])
    return hist.astype(float), centers.astype(float)

def _two_gauss_counts(x, A1, mu1, sig1, A2, mu2, sig2):
    sig1 = max(1e-12, float(sig1))
    sig2 = max(1e-12, float(sig2))
    return (A1 * np.exp(-0.5 * ((x - mu1) / sig1) ** 2)
            + A2 * np.exp(-0.5 * ((x - mu2) / sig2) ** 2))

def _smooth_counts(y, win=5):
    y = np.asarray(y, float)
    win = int(max(1, win))
    if win <= 1 or y.size < 3:
        return y
    k = np.ones(win, float) / win
    return np.convolve(y, k, mode="same")

def _pick_two_peaks(hist, centers, min_sep_bins=2):
    y = _smooth_counts(hist, win=5)
    x = np.asarray(centers, float)
    n = y.size
    if n < 3:
        return 0, max(0, n - 1)

    cand = []
    for i in range(1, n - 1):
        if y[i] >= y[i - 1] and y[i] >= y[i + 1]:
            cand.append(i)
    if not cand:
        cand = [int(np.argmax(y))]

    cand = sorted(cand, key=lambda i: y[i], reverse=True)
    i1 = cand[0]
    i2 = None
    for j in cand[1:]:
        if abs(int(j) - i1) >= int(min_sep_bins):
            i2 = int(j)
            break
    if i2 is None:
        i2 = 0 if i1 > n // 2 else (n - 1)
    return int(i1), int(i2)

def fit_two_gaussians_1d_counts(values, edges):
    v = np.asarray(values, float)
    v = v[np.isfinite(v)]
    if v.size < 6:
        return dict(ok=False, message="Not enough points (need >=6).")

    hist, centers = _hist_counts(v, edges)
    y = hist.astype(float)
    x = centers.astype(float)
    if np.all(y <= 0):
        return dict(ok=False, message="Empty histogram.")

    binw = float(edges[1] - edges[0]) if len(edges) >= 2 else 1.0
    xrng = float(edges[-1] - edges[0])

    sigma_min = max(0.35 * binw, 1e-3)
    sigma_max = max(0.25 * xrng, sigma_min * 3)
    A_min = 0.8
    delta_min = max(4.0 * binw, 0.20 * xrng)

    i1, i2 = _pick_two_peaks(y, x, min_sep_bins=2)
    mu1a, mu2a = float(x[i1]), float(x[i2])
    if mu2a < mu1a:
        mu1a, mu2a = mu2a, mu1a
        i1, i2 = i2, i1
    A1a, A2a = float(max(y[i1], 1.0)), float(max(y[i2], 1.0))

    sig0 = float(np.std(v)) if np.std(v) > 1e-12 else max(binw, 1.0)
    sig_init1 = float(np.clip(0.6 * sig0, sigma_min, sigma_max))
    sig_init2 = float(np.clip(0.6 * sig0, sigma_min, sigma_max))

    def _two_gauss_sep(xx, A1, mu1, sig1, A2, delta, sig2):
        mu2 = mu1 + max(delta_min, float(delta))
        return _two_gauss_counts(xx, A1, mu1, sig1, A2, mu2, sig2)

    p0 = [A1a, mu1a, sig_init1, A2a, max(delta_min, mu2a - mu1a), sig_init2]
    lo = [A_min, float(edges[0]), sigma_min, A_min, delta_min, sigma_min]
    hi = [np.inf, float(edges[-1]), sigma_max, np.inf, float(xrng), sigma_max]

    try:
        popt, _ = curve_fit(_two_gauss_sep, x, y, p0=p0, bounds=(lo, hi), maxfev=120000)
    except Exception as e:
        return dict(ok=False, message=f"Fit failed: {type(e).__name__}: {e}")

    A1, mu1, s1, A2, delta, s2 = map(float, popt)
    mu2 = mu1 + max(delta_min, float(delta))
    if mu2 < mu1:
        A1, A2 = A2, A1
        mu1, mu2 = mu2, mu1
        s1, s2 = s2, s1

    return dict(
        ok=True,
        message="OK",
        A1=A1, mu1=mu1, sig1=s1,
        A2=A2, mu2=mu2, sig2=s2,
        edges=edges.tolist(),
        centers=centers.tolist(),
        hist=hist.tolist(),
        n=int(v.size),
    )

def choose_free_combo_from_1d_means(fx, fy):
    mux = [float(fx["mu1"]), float(fx["mu2"])]
    muy = [float(fy["mu1"]), float(fy["mu2"])]
    best = None
    best_r = -1e99
    for i in range(2):
        for j in range(2):
            r = float(np.hypot(mux[i], muy[j]))
            if r > best_r:
                best_r = r
                best = (i, j)
    return best

def fit_gmm_2d_vxvy(vx_arr, vy_arr):
    """
    Fit a 2-component Gaussian Mixture Model directly in (vx, vy).

    Rule:
      - the component with smaller (mu_x + mu_y) is the left-bottom peak => free
      - the other component => trapped
    """
    vx_arr = np.asarray(vx_arr, float)
    vy_arr = np.asarray(vy_arr, float)

    m = np.isfinite(vx_arr) & np.isfinite(vy_arr)
    X = np.column_stack([vx_arr[m], vy_arr[m]])

    if X.shape[0] < 6:
        return dict(ok=False, message="Too few valid (vx, vy) points for 2D GMM.")

    try:
        gmm = GaussianMixture(
            n_components=2,
            covariance_type="diag",
            random_state=0,
            n_init=10,
            reg_covar=1e-6,
        )
        gmm.fit(X)
    except Exception as e:
        return dict(ok=False, message=f"GMM fit failed: {type(e).__name__}: {e}")

    means = np.asarray(gmm.means_, float)
    weights = np.asarray(gmm.weights_, float)

    covs_raw = np.asarray(gmm.covariances_, float)
    if gmm.covariance_type == "diag":
        covs = np.array([np.diag(c) for c in covs_raw], dtype=float)
    else:
        covs = covs_raw

    sums = means[:, 0] + means[:, 1]
    free_idx = int(np.argmin(sums))
    trapped_idx = 1 - free_idx

    labels_raw = gmm.predict(X)                   
    probs_raw = gmm.predict_proba(X)              


    labels_ft = np.where(labels_raw == free_idx, 0, 1)
    probs_ft = np.column_stack([
        probs_raw[:, free_idx],     
        probs_raw[:, trapped_idx],  
    ])

    return dict(
        ok=True,
        gmm=gmm,
        means=means,
        covs=covs,
        weights=weights,
        free_idx=free_idx,
        trapped_idx=trapped_idx,
        center_free=means[free_idx],
        center_trapped=means[trapped_idx],
        labels=labels_ft,
        probs=probs_ft,
        message="OK",
    )


def make_heatmap_gmm_2d(fit_pack, vx_range, vy_range):
    """
    Heatmap from the fitted 2D GMM density itself.
    """
    if not (isinstance(fit_pack, dict) and fit_pack.get("ok", False)):
        return None

    means = np.asarray(fit_pack.get("means", []), float)
    covs = np.asarray(fit_pack.get("covs", []), float)
    weights = np.asarray(fit_pack.get("weights", []), float)

    if means.shape != (2, 2) or covs.shape != (2, 2, 2) or weights.shape != (2,):
        return None

    vx0, vx1 = float(vx_range[0]), float(vx_range[1])
    vy0, vy1 = float(vy_range[0]), float(vy_range[1])

    xs = np.linspace(vx0, vx1, int(HEATMAP_NX))
    ys = np.linspace(vy0, vy1, int(HEATMAP_NY))
    Xg, Yg = np.meshgrid(xs, ys)

    P = np.zeros_like(Xg, dtype=float)

    for k in range(2):
        mu = means[k]
        cov = covs[k]

        try:
            cov_inv = np.linalg.inv(cov)
            det = float(np.linalg.det(cov))
        except Exception:
            continue

        if not np.isfinite(det) or det <= 0:
            continue

        DX = Xg - float(mu[0])
        DY = Yg - float(mu[1])

        quad = (
            cov_inv[0, 0] * DX * DX
            + 2.0 * cov_inv[0, 1] * DX * DY
            + cov_inv[1, 1] * DY * DY
        )

        norm = 1.0 / (2.0 * np.pi * np.sqrt(det))
        Zk = float(weights[k]) * norm * np.exp(-0.5 * quad)
        P += Zk

    pmax = float(np.nanmax(P)) if np.isfinite(np.nanmax(P)) else 0.0
    if pmax > 0:
        P = P / pmax

    return dict(xs=xs, ys=ys, Z=P)

def _gmm1d_from_2d_counts(x, weights, means, covs, n_total, binw, axis="vx"):
    """
    From fitted 2D GMM, build the expected 1D histogram COUNTS
    along vx or vy by taking the marginal of each 2D Gaussian.

    axis = "vx" or "vy"
    Returned y has the same meaning as histogram bin counts.
    """
    x = np.asarray(x, float)
    weights = np.asarray(weights, float)
    means = np.asarray(means, float)
    covs = np.asarray(covs, float)

    if means.shape != (2, 2) or covs.shape != (2, 2, 2) or weights.shape != (2,):
        return np.full_like(x, np.nan, dtype=float)

    ax = 0 if str(axis).lower() == "vx" else 1
    y = np.zeros_like(x, dtype=float)

    for k in range(2):
        mu = float(means[k, ax])
        sig2 = float(covs[k, ax, ax])
        if (not np.isfinite(sig2)) or sig2 <= 0:
            continue
        sig = np.sqrt(sig2)

        pdf = (1.0 / (np.sqrt(2.0 * np.pi) * sig)) * np.exp(-0.5 * ((x - mu) / sig) ** 2)


        y += float(weights[k]) * float(n_total) * float(binw) * pdf

    return y
# ============================================================
# ---------------- scanning input folder ----------------------
# ============================================================

def scan_videos_only_manual_xfm(base_dir):
    """
    Recursively scan all subfolders under base_dir and include:
      filtered_particle_tracks_v2__*__manually_added__linkfmt__subpixel__xfm.csv
    """
    tracks_files = []

    for root, _dirs, files in os.walk(base_dir):
        root_low = root.lower()
        if "_detections_out" not in root_low:
            continue
        if "varing laser power" not in root_low:
            continue

        for fn in files:
            fn_low = fn.lower()
            if not fn_low.endswith(".csv"):
                continue
            if not fn_low.startswith("filtered_particle_tracks_v2__"):
                continue
            if not fn_low.endswith(ONLY_SUFFIX.lower()):
                continue
            tracks_files.append(os.path.join(root, fn))

    tracks_files.sort()

    videos = {}
    for tr_path in tracks_files:
        stem = os.path.splitext(os.path.basename(tr_path))[0]
        folder = os.path.dirname(tr_path)

        tp_fn = "TrackParameters__from__" + stem + ".csv"
        tp_path = os.path.join(folder, tp_fn)

        # make key unique across different temperature folders
        parent_label = os.path.basename(os.path.dirname(folder))
        key = f"{parent_label} | {stem}"

        temp_k = extract_temp_k_from_key(parent_label)

        videos[key] = dict(
            key=key,
            tracks_csv=tr_path,
            tp_csv=(tp_path if os.path.isfile(tp_path) else ""),
            temp_k=temp_k,
        )

    return videos


# ============================================================
# ---------------------- data container -----------------------
# ============================================================

@dataclass
class VideoData:
    key: str
    tracks_csv: str
    original_tracks_csv: str
    deleted_tracks_csv: str
    tp_csv: str
    track_cache: dict
    t_end_max: float
    lum1: float

def build_all_combined_track_cache(video_data_dict):
    combined = {}
    new_tid = 1
    t_end_max_all = 0.0

    for vkey, vd in video_data_dict.items():
        if vkey == ALL_KEY:
            continue
        for old_tid in sorted(vd.track_cache.keys()):
            d0 = vd.track_cache[old_tid]
            d = dict(d0)
            d["tid"] = int(new_tid)
            d["unique_id"] = f"{vkey}__{d.get('unique_id', old_tid)}"
            combined[int(new_tid)] = d
            if np.isfinite(np.nanmax(d["t"])):
                t_end_max_all = max(t_end_max_all, float(np.nanmax(d["t"])))
            new_tid += 1

    return combined, float(t_end_max_all)
    
VIDEO_CONFIGS = scan_videos_only_manual_xfm(BASE_DIR)
if not VIDEO_CONFIGS:
    raise RuntimeError(
        f"No matching files found in:\n{BASE_DIR}\n"
        f"Need: filtered_particle_tracks_v2__*{ONLY_SUFFIX}"
    )

VIDEO_DATA = {}


for k, cfg in VIDEO_CONFIGS.items():
    original_csv = cfg["tracks_csv"]
    deleted_csv = build_deleted_output_path(original_csv)

    load_csv = deleted_csv if os.path.isfile(deleted_csv) else original_csv

    df_tr = pd.read_csv(load_csv)
    track_cache, t_end_max, lum1 = compute_track_level_metrics_from_tracks(df_tr, CM_PER_PIXEL, FPS)

    VIDEO_DATA[k] = VideoData(
        key=k,
        tracks_csv=load_csv,
        original_tracks_csv=original_csv,
        deleted_tracks_csv=deleted_csv,
        tp_csv=cfg.get("tp_csv", ""),
        track_cache=track_cache,
        t_end_max=t_end_max,
        lum1=lum1,
    )
track_cache_all, t_end_max_all = build_all_combined_track_cache(VIDEO_DATA)
VIDEO_DATA[ALL_KEY] = VideoData(
    key=ALL_KEY,
    tracks_csv="(combined) " + BASE_DIR,
    original_tracks_csv="(combined)",
    deleted_tracks_csv="(combined)",
    tp_csv="(not used)",
    track_cache=track_cache_all,
    t_end_max=t_end_max_all,
    lum1=np.nan,
)


VIDEO_KEYS = list(VIDEO_DATA.keys())

REFERENCE_VIDEO_KEY = [k for k in VIDEO_KEYS if k != ALL_KEY][0]

def get_unique_id_from_tid(vkey, tid):
    vd = VIDEO_DATA[vkey]
    d0 = vd.track_cache.get(int(tid))
    if d0 is None:
        return None
    return str(d0.get("unique_id", ""))


def delete_track_and_save_csv(vkey, tid):
    """
    Delete the selected track completely from the preferred input csv
    and save to deleted_tracks_csv. Future runs will prefer that file.
    """
    vd = VIDEO_DATA[vkey]
    uid_to_delete = get_unique_id_from_tid(vkey, tid)
    if uid_to_delete is None:
        return False, f"Track id {tid} not found."

    load_csv = vd.tracks_csv
    out_csv = vd.deleted_tracks_csv

    try:
        df = pd.read_csv(load_csv)
    except Exception as e:
        return False, f"Failed reading csv: {e}"

    uid_col = _pick_existing_col(df, ["unique_id", "Unique_ID"])
    if uid_col is None:
        return False, f"CSV missing unique_id column. Columns: {list(df.columns)}"

    n_before = len(df)
    m_keep = df[uid_col].astype(str) != str(uid_to_delete)
    df_new = df.loc[m_keep].copy()
    n_after = len(df_new)

    removed_rows = int(n_before - n_after)
    if removed_rows <= 0:
        return False, f"No rows removed for unique_id={uid_to_delete}"

    try:
        df_new.to_csv(out_csv, index=False)
    except Exception as e:
        return False, f"Failed writing csv: {e}"

    return True, f"Deleted tid={tid}, unique_id={uid_to_delete}, removed_rows={removed_rows}, saved -> {out_csv}"

# ============================================================
# ---------------- selection + effective track ----------------
# ============================================================

def _split_until_first_gap(t, x, y, frames):
    """
    Keep only the first continuous segment.
    Once a frame gap (>1) appears, permanently stop.
    """
    t = np.asarray(t, float)
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    frames = np.asarray(frames, int)

    n = len(frames)
    if n == 0:
        return (
            np.asarray([], float),
            np.asarray([], float),
            np.asarray([], float),
            np.asarray([], int),
        )

    cut = n
    for i in range(n - 1):
        if int(frames[i + 1]) - int(frames[i]) > 1:
            cut = i + 1
            break

    return t[:cut], x[:cut], y[:cut], frames[:cut]


def _trim_by_t_with_frames(t, x, y, frames, t_min):
    t = np.asarray(t, float)
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    frames = np.asarray(frames, int)

    m = np.isfinite(t) & np.isfinite(x) & np.isfinite(y) & (t >= float(t_min))
    if not np.any(m):
        return (
            np.asarray([], float),
            np.asarray([], float),
            np.asarray([], float),
            np.asarray([], int),
        )

    return t[m], x[m], y[m], frames[m]

def descending_stage_info(track_dict, t_min, min_tail_points=3):
    t = np.asarray(track_dict["t"], float)
    y = np.asarray(track_dict["y"], float)
    frame_all = np.rint(t * float(FPS)).astype(int)

    dummy_x = np.zeros_like(y, dtype=float)

    t2, _x2, y2, f2 = _trim_by_t_with_frames(
        t, dummy_x, y, frame_all, float(t_min)
    )
    t2, _x2, y2, f2 = _split_until_first_gap(
        t2, dummy_x[:len(t2)], y2, f2
    )

    if len(y2) < (min_tail_points + 1):
        return dict(
            reached_descending_stage=False,
            enough_tail_after_ypeak=False,
            tail_pts_after_ypeak=0,
        )

    try:
        i_peak = int(np.nanargmax(y2))
    except Exception:
        return dict(
            reached_descending_stage=False,
            enough_tail_after_ypeak=False,
            tail_pts_after_ypeak=0,
        )

    tail_n = int(len(y2) - (i_peak + 1))
    if tail_n < int(min_tail_points):
        return dict(
            reached_descending_stage=False,
            enough_tail_after_ypeak=False,
            tail_pts_after_ypeak=tail_n,
        )

    y_tail = y2[i_peak + 1:]
    t_tail = t2[i_peak + 1:]

    if len(y_tail) < int(min_tail_points):
        return dict(
            reached_descending_stage=False,
            enough_tail_after_ypeak=False,
            tail_pts_after_ypeak=int(len(y_tail)),
        )

    try:
        slope = float(np.polyfit(t_tail, y_tail, 1)[0])
    except Exception:
        return dict(
            reached_descending_stage=False,
            enough_tail_after_ypeak=True,
            tail_pts_after_ypeak=int(len(y_tail)),
        )

    return dict(
        reached_descending_stage=bool(np.isfinite(slope) and (slope < 0.0)),
        enough_tail_after_ypeak=True,
        tail_pts_after_ypeak=int(len(y_tail)),
    )

def effective_track(vkey, tid, t_min, cluster_n_threshold):
    vd = VIDEO_DATA[vkey]
    d0 = vd.track_cache[int(tid)]
    d = dict(d0)
    n_est_now = d.get("n_est", np.nan)
    if np.isfinite(n_est_now):
        d["is_cluster"] = bool(float(n_est_now) >= float(cluster_n_threshold))
    else:
        d["is_cluster"] = False

    t_all = np.asarray(d["t"], float)
    x_all = np.asarray(d["x"], float)
    y_all = np.asarray(d["y"], float)


    frame_all = np.rint(t_all * float(FPS)).astype(int)


    t_post, x_post, y_post, frame_post = _trim_by_t_with_frames(
        t_all, x_all, y_all, frame_all, float(t_min)
    )


    t_plot, x_plot, y_plot, frame_plot = _split_until_first_gap(
        t_post, x_post, y_post, frame_post
    )

    d["t_plot"] = t_plot
    d["x_plot"] = x_plot
    d["y_plot"] = y_plot
    d["frame_plot"] = frame_plot

    d["has_post_tmin"] = bool(t_plot.size >= 2)
    d["n_points_post_tmin"] = int(t_plot.size)

    desc_info = descending_stage_info(
        d, t_min=float(t_min), min_tail_points=3
    )
    d["reached_descending_stage"] = bool(desc_info["reached_descending_stage"])
    d["enough_tail_after_ypeak"] = bool(desc_info["enough_tail_after_ypeak"])
    d["tail_pts_after_ypeak"] = int(desc_info["tail_pts_after_ypeak"])

    vv = terminal_v_quadratic_with_fallback(d, t_min=float(t_min))
    if vv is None:
        d["vx"] = np.nan
        d["vy"] = np.nan
        d["speed"] = np.nan
    else:
        d["vx"] = vv["vx"]
        d["vy"] = vv["vy"]
        d["speed"] = vv["speed"]

    return d

def compute_fit_pack(
    vkey,
    t_min,
    min_points,
    cluster_n_threshold,
    free_s_threshold,
    include_clusters_in_fit=True,
):
    vd = VIDEO_DATA[vkey]

    vx_list, vy_list, used = [], [], []
    for tid in sorted(vd.track_cache.keys()):
        d = effective_track(vkey, tid, t_min, cluster_n_threshold)

        if HIDE_TRACKS_WITHOUT_POST_TMIN and (not d.get("has_post_tmin", False)):
            continue

        if int(d.get("n_points_post_tmin", 0)) < int(min_points):
            continue

        if not bool(d.get("enough_tail_after_ypeak", False)):
            continue


        if (not include_clusters_in_fit) and bool(d.get("is_cluster", False)):
            continue


        if not bool(d.get("reached_descending_stage", False)):
            continue
        
        s_full = d.get("s_full", np.nan)
        if (not np.isfinite(s_full)) or (float(s_full) < float(free_s_threshold)):
            continue
        
        if np.isfinite(d.get("vx", np.nan)) and np.isfinite(d.get("vy", np.nan)):
            vx_list.append(float(d["vx"]))
            vy_list.append(float(d["vy"]))
            used.append(int(tid))

    vx_arr = np.asarray(vx_list, float)
    vy_arr = np.asarray(vy_list, float)

    if vx_arr.size < 6:
        return dict(
            ok=False,
            message=(
                f"Fit sample too small after trajectory-quality cuts: "
                f"n={int(vx_arr.size)} tracks. "
                f"include_clusters_in_fit={include_clusters_in_fit}"
            )
        )


    edges_vx = _hist_edges_from_values(vx_arr, bins=HIST_BINS)
    edges_vy = _hist_edges_from_values(vy_arr, bins=HIST_BINS)

    fx = fit_two_gaussians_1d_counts(vx_arr, edges_vx)
    fy = fit_two_gaussians_1d_counts(vy_arr, edges_vy)
    if (not fx.get("ok")) or (not fy.get("ok")):
        return dict(
            ok=False,
            message=f"Fit failed.\nVx: {fx.get('message')}\nVy: {fy.get('message')}"
        )


    g2 = fit_gmm_2d_vxvy(vx_arr, vy_arr)
    if not g2.get("ok", False):
        return dict(ok=False, message=g2.get("message", "2D GMM failed."))

    return dict(
        ok=True,
        n_used=int(len(used)),
        used_ids=used,
        vx_fit=vx_arr.tolist(),
        vy_fit=vy_arr.tolist(),

        fx=fx,
        fy=fy,

        means=np.asarray(g2["means"], float).tolist(),
        covs=np.asarray(g2["covs"], float).tolist(),
        weights=np.asarray(g2["weights"], float).tolist(),

        free_idx=int(g2["free_idx"]),
        trapped_idx=int(g2["trapped_idx"]),

        center_free=np.asarray(g2["center_free"], float).tolist(),
        center_trapped=np.asarray(g2["center_trapped"], float).tolist(),

        labels_2d=np.asarray(g2["labels"], int).tolist(),
        probs_2d=np.asarray(g2["probs"], float).tolist(),

        free_name="left-bottom GMM component",
        trapped_name="right-top GMM component",
    )


def auto_classify_state2(d, fit_pack):
    """
    Classification using the fitted 2D GMM in (vx, vy).

    free    = component at lower-left
    trapped = component at upper-right
    """
    if not (isinstance(fit_pack, dict) and fit_pack.get("ok", False)):
        return "trapped"

    vx = d.get("vx", np.nan)
    vy = d.get("vy", np.nan)
    if not (np.isfinite(vx) and np.isfinite(vy)):
        return "trapped"

    means = np.asarray(fit_pack.get("means", []), float)
    covs = np.asarray(fit_pack.get("covs", []), float)
    weights = np.asarray(fit_pack.get("weights", []), float)
    free_idx = int(fit_pack.get("free_idx", 0))
    trapped_idx = int(fit_pack.get("trapped_idx", 1))

    if means.shape != (2, 2) or covs.shape != (2, 2, 2) or weights.shape != (2,):
        return "trapped"

    pt = np.array([float(vx), float(vy)], dtype=float)

    logp = []
    for k in range(2):
        mu = means[k]
        cov = covs[k]

        try:
            cov_inv = np.linalg.inv(cov)
            det = float(np.linalg.det(cov))
        except Exception:
            return "trapped"

        if (not np.isfinite(det)) or det <= 0:
            return "trapped"

        diff = pt - mu
        quad = float(diff.T @ cov_inv @ diff)
        lp = np.log(max(float(weights[k]), 1e-300)) - 0.5 * (
            2.0 * np.log(2.0 * np.pi) + np.log(det) + quad
        )
        logp.append(lp)

    pred = int(np.argmax(logp))
    return "free" if pred == free_idx else "trapped"


# ============================================================
# ---------------- heatmap (product P(Vx)*P(Vy)) --------------
# ============================================================



def build_vx_peak_vs_laser_figure_alltemps(
    t_min,
    min_points,
    cluster_n_threshold,
    free_s_threshold,
    include_clusters_in_fit=True,
    peak_which=1,
):
    groups = {}

    color_seq = [
        "#636EFA",
        "#00CC96",
        "#FFA15A",
        "#FF6692",
        "#AB63FA",
        "#19D3F3",
        "#B6E880",
    ]

    for vkey in VIDEO_KEYS:
        if vkey == ALL_KEY:
            continue

        us = extract_laser_us_from_key(vkey)
        power_mw = laser_us_to_power_mw(us)
        temp_k_raw = extract_temp_k_from_key(vkey)

        if not np.isfinite(power_mw):
            continue

        fit_pack = compute_fit_pack(
            vkey,
            t_min,
            min_points,
            cluster_n_threshold,
            free_s_threshold,
            include_clusters_in_fit=include_clusters_in_fit,
        )
        if not (isinstance(fit_pack, dict) and fit_pack.get("ok", False)):
            continue

        means = np.asarray(fit_pack["means"], float)
        covs = np.asarray(fit_pack["covs"], float)
        free_idx = int(fit_pack["free_idx"])
        other_idx = 1 - free_idx

        idx = free_idx if int(peak_which) == 1 else other_idx

        vx_abs = abs(float(means[idx, 0]))
        sx = np.sqrt(max(float(covs[idx, 0, 0]), 0.0))

        if not (np.isfinite(vx_abs) and np.isfinite(sx)):
            continue

        temp_k = canonical_temp_k_for_grouping(temp_k_raw)

        if np.isfinite(temp_k):
            temp_label = f"{temp_k:.3f} K"
        else:
            temp_label = "unknown T"

        if temp_label not in groups:
            groups[temp_label] = {
                "x": [],
                "y": [],
                "e": [],
                "text": [],
            }

        groups[temp_label]["x"].append(power_mw)
        groups[temp_label]["y"].append(vx_abs)
        groups[temp_label]["e"].append(sx)
        groups[temp_label]["text"].append(vkey)

    fig = go.Figure()

    for i, (temp_label, g) in enumerate(sorted(groups.items())):
        xs = np.asarray(g["x"], float)
        ys = np.asarray(g["y"], float)
        es = np.asarray(g["e"], float)
        texts = list(g["text"])

        if xs.size == 0:
            continue

        order = np.argsort(xs)
        xs = xs[order]
        ys = ys[order]
        es = es[order]
        texts = [texts[j] for j in order]

        col = color_seq[i % len(color_seq)]

        fig.add_trace(go.Scatter(
            x=xs,
            y=ys,
            mode="markers",
            marker=dict(size=8, symbol="circle", color=col),
            error_y=dict(
                type="data",
                array=es,
                visible=True,
                thickness=1.3,
                width=4,
                color=col
            ),
            text=texts,
            hovertemplate=(
                "video=%{text}<br>"
                f"T={temp_label}<br>"
                "laser power=%{x:.4g} mW<br>"
                f"|Vx_{peak_which}|=%{{y:.4g}} cm/s<br>"
                "sigma_x=%{error_y.array:.4g} cm/s"
                "<extra></extra>"
            ),
            name=temp_label,
            showlegend=True,
            legendgroup=temp_label,
        ))

    fig.update_layout(
        height=520,
        margin=dict(l=60, r=20, t=70, b=55),
        title=dict(
            text=f"All temperatures: Vx_{peak_which} vs laser power",
            x=0.5,
            xanchor="center",
            y=0.92,
        ),
        hovermode="closest",
        autosize=True,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.0,
            xanchor="left",
            x=0.0,
            font=dict(size=9),
            bgcolor="rgba(255,255,255,0.65)",
            traceorder="normal",
        ),
    )
    fig.update_xaxes(title_text="laser power (mW)", automargin=True)
    fig.update_yaxes(title_text=f"|Vx_{peak_which}| (cm/s)", automargin=True)

    return fig

def build_vx1_vs_laser_figure_alltemps(
    t_min,
    min_points,
    cluster_n_threshold,
    free_s_threshold,
    include_clusters_in_fit=True,
):
    return build_vx_peak_vs_laser_figure_alltemps(
        t_min=t_min,
        min_points=min_points,
        cluster_n_threshold=cluster_n_threshold,
        free_s_threshold=free_s_threshold,
        include_clusters_in_fit=include_clusters_in_fit,
        peak_which=1,
    )


def build_vx2_vs_laser_figure_alltemps(
    t_min,
    min_points,
    cluster_n_threshold,
    free_s_threshold,
    include_clusters_in_fit=True,
):
    return build_vx_peak_vs_laser_figure_alltemps(
        t_min=t_min,
        min_points=min_points,
        cluster_n_threshold=cluster_n_threshold,
        free_s_threshold=free_s_threshold,
        include_clusters_in_fit=include_clusters_in_fit,
        peak_which=2,
    )


def add_current_temp_vx12_subplot(
    fig,
    vkey,
    t_min,
    min_points,
    cluster_n_threshold,
    free_s_threshold,
    include_clusters_in_fit=True,
    row=3,
    col=3,
):
    temp_ref = extract_temp_k_from_key(vkey.split("|")[0])

    xs1, ys1, es1 = [], [], []
    xs2, ys2, es2 = [], [], []

    for vk in VIDEO_KEYS:
        if vk == ALL_KEY:
            continue

        temp_k = extract_temp_k_from_key(vk.split("|")[0])
        if np.isfinite(temp_ref) and np.isfinite(temp_k):
            if abs(temp_k - temp_ref) > 1e-9:
                continue
        else:
            continue

        us = extract_laser_us_from_key(vk)
        power_mw = laser_us_to_power_mw(us)
        if not np.isfinite(power_mw):
            continue

        fit_pack = compute_fit_pack(
            vk,
            t_min,
            min_points,
            cluster_n_threshold,
            free_s_threshold,
            include_clusters_in_fit=include_clusters_in_fit,
        )
        if not (isinstance(fit_pack, dict) and fit_pack.get("ok", False)):
            continue

        means = np.asarray(fit_pack["means"], float)
        covs = np.asarray(fit_pack["covs"], float)
        free_idx = int(fit_pack["free_idx"])
        other_idx = 1 - free_idx

        vx1 = abs(float(means[free_idx, 0]))
        sx1 = np.sqrt(max(float(covs[free_idx, 0, 0]), 0.0))

        vx2 = abs(float(means[other_idx, 0]))
        sx2 = np.sqrt(max(float(covs[other_idx, 0, 0]), 0.0))

        if np.isfinite(vx1) and np.isfinite(sx1):
            xs1.append(power_mw)
            ys1.append(vx1)
            es1.append(sx1)

        if np.isfinite(vx2) and np.isfinite(sx2):
            xs2.append(power_mw)
            ys2.append(vx2)
            es2.append(sx2)

    color_vx1 = "red"
    color_vx2 = "green"

    if len(xs1) > 0:
        order = np.argsort(xs1)
        xs1 = np.asarray(xs1, float)[order]
        ys1 = np.asarray(ys1, float)[order]
        es1 = np.asarray(es1, float)[order]

        fig.add_trace(go.Scatter(
            x=xs1,
            y=ys1,
            mode="markers",
            marker=dict(size=8, symbol="circle", color=color_vx1),
            error_y=dict(
                type="data",
                array=es1,
                visible=True,
                thickness=1.2,
                width=3,
                color=color_vx1
            ),
            name="Vx1 data",
            showlegend=False,
            legendgroup="currentT_vx1",
            hovertemplate=(
                "laser power=%{x:.4g} mW<br>"
                "|Vx1|=%{y:.4g} cm/s<br>"
                "sigma_x=%{error_y.array:.4g} cm/s"
                "<extra></extra>"
            ),
        ), row=row, col=col)

    if len(xs2) > 0:
        order = np.argsort(xs2)
        xs2 = np.asarray(xs2, float)[order]
        ys2 = np.asarray(ys2, float)[order]
        es2 = np.asarray(es2, float)[order]

        fig.add_trace(go.Scatter(
            x=xs2,
            y=ys2,
            mode="markers",
            marker=dict(size=8, symbol="circle", color=color_vx2),
            error_y=dict(
                type="data",
                array=es2,
                visible=True,
                thickness=1.2,
                width=3,
                color=color_vx2
            ),
            name="Vx2 data",
            showlegend=False,
            legendgroup="currentT_vx2",
            hovertemplate=(
                "laser power=%{x:.4g} mW<br>"
                "|Vx2|=%{y:.4g} cm/s<br>"
                "sigma_x=%{error_y.array:.4g} cm/s"
                "<extra></extra>"
            ),
        ), row=row, col=col)

    fig.update_xaxes(title_text="laser power (mW)", row=row, col=col)
    fig.update_yaxes(title_text="|Vx| (cm/s)", row=row, col=col)

    
# ============================================================
# ---------------------- figure builder -----------------------
# ============================================================

def class3_style(cls):
    if cls == "free":
        return COL_FREE, ALPHA_FREE
    if cls == "cluster":
        return COL_CLUSTER, ALPHA_CLUSTER
    return COL_TRAPPED, ALPHA_TRAPPED


def final_class3(d, fit_pack, free_s_threshold):
    if bool(d.get("is_cluster", False)):
        return "cluster"

    if not bool(d.get("reached_descending_stage", False)):
        return "trapped"

    s_full = d.get("s_full", np.nan)
    if (not np.isfinite(s_full)) or (float(s_full) < float(free_s_threshold)):
        return "trapped"

    return auto_classify_state2(d, fit_pack)

def explain_selected_class(d, fit_pack, min_points, free_s_threshold):
    if bool(d.get("is_cluster", False)):
        return "cluster", "cluster because N_est >= cluster threshold"

    if not bool(d.get("has_post_tmin", False)):
        return "hidden", "hidden because no valid post-t_min segment"

    if int(d.get("n_points_post_tmin", 0)) < int(min_points):
        return "hidden", "hidden because too few points after t_min"

    if not bool(d.get("enough_tail_after_ypeak", False)):
        return "hidden", "hidden because too few points after y-peak"

    if not bool(d.get("reached_descending_stage", False)):
        return "trapped", "trapped because tail after y-peak is not descending"
    
    s_full = d.get("s_full", np.nan)
    if (not np.isfinite(s_full)) or (float(s_full) < float(free_s_threshold)):
        return "trapped", f"trapped because s < {float(free_s_threshold):.4g}"

    vx = d.get("vx", np.nan)
    vy = d.get("vy", np.nan)
    if not (np.isfinite(vx) and np.isfinite(vy)):
        return "hidden", "hidden because terminal velocity could not be computed"

    if not (isinstance(fit_pack, dict) and fit_pack.get("ok", False)):
        return "trapped", "trapped because fit_pack is unavailable"

    cls2 = auto_classify_state2(d, fit_pack)
    if cls2 == "free":
        return "free", "classified as free by 2D GMM"
    return "trapped", "classified as trapped by 2D GMM"
    
def _add_clickable_line(fig, *, x, y, tid, row, col):
    fig.add_trace(
        go.Scatter(
            x=x, y=y,
            mode="lines",
            line=dict(color="rgba(0,0,0,0)", width=HITBOX_WIDTH),
            customdata=[int(tid)] * len(x),
            hoverinfo="none",
            showlegend=False
        ),
        row=row, col=col
    )

def _line_until_first_gap(x, y, frames):
    """
    Draw line only up to the first frame gap.
    After the first gap, do NOT connect later points.

    gap rule:
        consecutive frames must satisfy df <= 1
    """
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    frames = np.asarray(frames, int)

    n = len(frames)
    if n <= 1:
        return x, y

    cut = n
    for i in range(n - 1):
        if int(frames[i + 1]) - int(frames[i]) > 1:
            cut = i + 1
            break

    return x[:cut], y[:cut]

def relaxation_time_from_free_gmm(fit_pack):
    """
    Calculate apparent relaxation time from the left-bottom/free GMM component.

    Uses:
        tau = |mu_y| / g'

    where:
        mu_y is in cm/s
        g' is in cm/s^2

    The uncertainty is estimated from the GMM vertical width:
        sigma_tau = sigma_y / g'

    Returns tau in seconds and milliseconds.
    """
    if not (isinstance(fit_pack, dict) and fit_pack.get("ok", False)):
        return dict(ok=False, message="fit_pack is not valid.")

    means = np.asarray(fit_pack.get("means", []), float)
    covs = np.asarray(fit_pack.get("covs", []), float)
    weights = np.asarray(fit_pack.get("weights", []), float)
    free_idx = int(fit_pack.get("free_idx", 0))

    if means.shape != (2, 2) or covs.shape != (2, 2, 2):
        return dict(ok=False, message="GMM means/covs have unexpected shape.")

    mu_y = float(means[free_idx, 1])
    sigma_y = float(np.sqrt(max(float(covs[free_idx, 1, 1]), 0.0)))

    if not (np.isfinite(mu_y) and np.isfinite(sigma_y)):
        return dict(ok=False, message="mu_y or sigma_y is not finite.")

    tau_s = abs(mu_y) / float(G_PRIME_CM_S2)
    tau_err_s = sigma_y / float(G_PRIME_CM_S2)

    n_used = int(fit_pack.get("n_used", 0))
    weight_free = float(weights[free_idx]) if weights.shape == (2,) else np.nan
    n_eff = float(weight_free * n_used) if np.isfinite(weight_free) else np.nan

    if np.isfinite(n_eff) and n_eff > 1:
        tau_sem_s = tau_err_s / np.sqrt(n_eff)
    else:
        tau_sem_s = np.nan

    return dict(
        ok=True,
        free_idx=free_idx,
        mu_y_cm_s=mu_y,
        sigma_y_cm_s=sigma_y,
        tau_s=tau_s,
        tau_ms=1000.0 * tau_s,
        tau_err_s=tau_err_s,
        tau_err_ms=1000.0 * tau_err_s,
        tau_sem_s=tau_sem_s,
        tau_sem_ms=1000.0 * tau_sem_s if np.isfinite(tau_sem_s) else np.nan,
        n_used=n_used,
        weight_free=weight_free,
        n_eff=n_eff,
    )
    
def build_figure(
    vkey,
    selected_tid,
    show_motion,
    show_size,
    t_min,
    min_points,
    cluster_n_threshold,
    free_s_threshold,
    include_clusters_in_fit,
    show_heatmap,
):
    vd = VIDEO_DATA[vkey]
    fit_pack = compute_fit_pack(
        vkey,
        t_min,
        min_points,
        cluster_n_threshold,
        free_s_threshold,
        include_clusters_in_fit=include_clusters_in_fit,
    )

    vis = []
    for tid in sorted(vd.track_cache.keys()):
        d = effective_track(vkey, tid, t_min, cluster_n_threshold)

        if HIDE_TRACKS_WITHOUT_POST_TMIN and (not d["has_post_tmin"]):
            continue

        if int(d.get("n_points_post_tmin", 0)) < int(min_points):
            continue

        if not bool(d.get("enough_tail_after_ypeak", False)):
            continue

        cls = final_class3(d, fit_pack, free_s_threshold)
        is_cluster = (cls == "cluster")

        if cls in ("free", "trapped") and (cls not in show_motion):
            continue

        if is_cluster and ("cluster" not in show_size):
            continue

        if (not is_cluster) and ("single" not in show_size):
            continue

        vis.append(int(tid))

    titles = [
        "w–s",
        f"XY (cm), t ≥ {t_min:g}s",
        f"v_y vs v_x (cm/s), t ≥ {t_min:g}s",
        "hist v_x",
        "x(t)",
        "hist speed",
        "hist v_y",
        "y(t)",
        "current T: Vx1, Vx2 vs laser power",
        "hist estimated particle number",
    ]

    fig = make_subplots(
        rows=3, cols=4,
        specs=[
            [{}, {"rowspan": 3}, {}, {}],
            [{}, None,           {}, {}],
            [{}, None,           {}, {}],
        ],
        subplot_titles=titles,
        horizontal_spacing=0.06,
        vertical_spacing=0.14,
        row_heights=[0.34, 0.33, 0.33],
        column_widths=[0.25, 0.33, 0.21, 0.21],
    )

    # WS points
    for is_cluster, symbol, ms in [(False, "circle", 7), (True, "diamond", 8)]:
        xs, ys, cols, tids = [], [], [], []
        for tid in vis:
            d = effective_track(vkey, tid, t_min, cluster_n_threshold)
            if bool(d["is_cluster"]) != bool(is_cluster):
                continue
            if not (np.isfinite(d.get("s_full", np.nan)) and np.isfinite(d.get("w", np.nan))):
                continue

            cls = final_class3(d, fit_pack, free_s_threshold)
            c, a = class3_style(cls)

            xs.append(float(d["s_full"]))
            ys.append(float(d["w"]))
            cols.append(c)
            tids.append(int(tid))

        if xs:
            fig.add_trace(
                go.Scatter(
                    x=xs,
                    y=ys,
                    mode="markers",
                    marker=dict(size=ms, color=cols, symbol=symbol, opacity=0.92),
                    customdata=tids,
                    hovertext=[f"tid={t}" for t in tids],
                    hoverinfo="text",
                    showlegend=False,
                ),
                row=1,
                col=1
            )

    # XY + xt/yt
    for tid in vis:
        d = effective_track(vkey, tid, t_min, cluster_n_threshold)

        cls = final_class3(d, fit_pack, free_s_threshold)
        line_col, a = class3_style(cls)

        line_width = 2.0
        marker_symbol = "diamond" if cls == "cluster" else "circle"
        marker_size = 4 if cls == "cluster" else 3

        fig.add_trace(
            go.Scatter(
                x=d["x_plot"],
                y=d["y_plot"],
                mode="lines+markers",
                line=dict(color=line_col, width=line_width),
                marker=dict(
                    color=line_col,
                    symbol=marker_symbol,
                    size=marker_size,
                ),
                opacity=a,
                hoverinfo="skip",
                showlegend=False,
            ),
            row=1,
            col=2
        )
        if len(d["x_plot"]) >= 2:
            _add_clickable_line(fig, x=d["x_plot"], y=d["y_plot"], tid=tid, row=1, col=2)

        fig.add_trace(
            go.Scatter(
                x=d["t_plot"],
                y=d["x_plot"],
                mode="lines+markers",
                line=dict(color=line_col, width=line_width),
                marker=dict(
                    color=line_col,
                    symbol=marker_symbol,
                    size=marker_size,
                ),
                opacity=a,
                hoverinfo="skip",
                showlegend=False,
            ),
            row=2,
            col=1
        )
        if len(d["t_plot"]) >= 2:
            _add_clickable_line(fig, x=d["t_plot"], y=d["x_plot"], tid=tid, row=2, col=1)

        fig.add_trace(
            go.Scatter(
                x=d["t_plot"],
                y=d["y_plot"],
                mode="lines+markers",
                line=dict(color=line_col, width=line_width),
                marker=dict(
                    color=line_col,
                    symbol=marker_symbol,
                    size=marker_size,
                ),
                opacity=a,
                hoverinfo="skip",
                showlegend=False,
            ),
            row=3,
            col=1
        )
        if len(d["t_plot"]) >= 2:
            _add_clickable_line(fig, x=d["t_plot"], y=d["y_plot"], tid=tid, row=3, col=1)
            
    # Vy vs Vx
    vx_all, vy_all = [], []
    for tid in vis:
        d = effective_track(vkey, tid, t_min, cluster_n_threshold)
        if np.isfinite(d.get("vx", np.nan)) and np.isfinite(d.get("vy", np.nan)):
            vx_all.append(float(d["vx"]))
            vy_all.append(float(d["vy"]))
    vx_rng = _auto_range(vx_all, pad_frac=0.10, fallback=(-1, 1))
    vy_rng = _auto_range(vy_all, pad_frac=0.10, fallback=(-1, 1))

    if show_heatmap and fit_pack.get("ok", False):
        hm = make_heatmap_gmm_2d(fit_pack, vx_rng, vy_rng)
        if hm is not None:
            fig.add_trace(go.Heatmap(
                x=hm["xs"], y=hm["ys"], z=hm["Z"],
                zmin=0.0, zmax=1.0,
                colorscale="Turbo",
                opacity=0.55,
                showscale=True,
                colorbar=dict(title="2D GMM\n(norm.)", len=0.75, thickness=14, x=1.02),
                hoverinfo="skip",
            ), row=1, col=3)

    for is_cluster, symbol in [(False, "circle"), (True, "diamond")]:
        xs, ys, cols, tids = [], [], [], []
        for tid in vis:
            d = effective_track(vkey, tid, t_min, cluster_n_threshold)
            if bool(d["is_cluster"]) != bool(is_cluster):
                continue
            if not (np.isfinite(d.get("vx", np.nan)) and np.isfinite(d.get("vy", np.nan))):
                continue

            cls = final_class3(d, fit_pack, free_s_threshold)
            c, a = class3_style(cls)

            xs.append(float(d["vx"]))
            ys.append(float(d["vy"]))
            cols.append(c)
            tids.append(int(tid))

        if xs:
            fig.add_trace(
                go.Scatter(
                    x=xs,
                    y=ys,
                    mode="markers",
                    marker=dict(size=9, color=cols, symbol=symbol, opacity=0.85, line=dict(width=0)),
                    customdata=tids,
                    hovertext=[f"tid={t}" for t in tids],
                    hoverinfo="text",
                    showlegend=False,
                ),
                row=1,
                col=3
            )

    def add_hist(var_key, row, col, xlabel):
        vals_free = []
        vals_tr = []
        vals_cluster = []

        ids_free = []
        ids_tr = []
        ids_cluster = []

        for tid in vis:
            d = effective_track(vkey, tid, t_min, cluster_n_threshold)
            v = d.get(var_key, np.nan)
            if not np.isfinite(v):
                continue

            cls = final_class3(d, fit_pack, free_s_threshold)

            if cls == "free":
                vals_free.append(float(v))
                ids_free.append(int(tid))
            elif cls == "trapped":
                vals_tr.append(float(v))
                ids_tr.append(int(tid))
            else:
                vals_cluster.append(float(v))
                ids_cluster.append(int(tid))

        all_vals = vals_free + vals_tr + vals_cluster

        if var_key == "n_est":
            all_vals_arr = np.asarray(all_vals, float)
            all_vals_arr = all_vals_arr[np.isfinite(all_vals_arr)]

            visible_vals = all_vals_arr[all_vals_arr <= float(N_EST_XMAX)]

            if visible_vals.size > 0:
                n_min = float(np.nanmin(visible_vals))
            elif all_vals_arr.size > 0:
                n_min = float(np.nanmin(all_vals_arr))
            else:
                n_min = 0.0

            if n_min >= float(N_EST_XMAX):
                n_min = float(N_EST_XMAX) - 1.0

            edges = np.linspace(
                n_min,
                float(N_EST_XMAX),
                int(N_EST_HIST_BINS) + 1
            )
        else:
            edges = _hist_edges_from_values(all_vals, bins=HIST_BINS)

        centers = 0.5 * (edges[:-1] + edges[1:])
        binw = float(edges[1] - edges[0]) if len(edges) >= 2 else 1.0

        def bin_ids(vals, ids):
            out = [[] for _ in range(len(centers))]
            for v, tid in zip(vals, ids):
                bi = int(np.searchsorted(edges, v, side="right") - 1)
                if 0 <= bi < len(centers):
                    out[bi].append(int(tid))
            return out

        tr_bins = bin_ids(vals_tr, ids_tr)
        cl_bins = bin_ids(vals_cluster, ids_cluster)
        fr_bins = bin_ids(vals_free, ids_free)

        tr_counts = np.array([len(b) for b in tr_bins], int)
        cl_counts = np.array([len(b) for b in cl_bins], int)
        fr_counts = np.array([len(b) for b in fr_bins], int)

        fig.add_trace(
            go.Bar(
                x=centers,
                y=tr_counts,
                width=binw * 0.95,
                marker=dict(color=COL_TRAPPED),
                opacity=ALPHA_TRAPPED,
                customdata=[{"kind": "hist_bin", "ids": b} for b in tr_bins],
                hovertext=[f"trapped bin ids: {b[:12]}{'...' if len(b) > 12 else ''}" for b in tr_bins],
                hoverinfo="text",
                showlegend=False,
            ),
            row=row,
            col=col
        )

        fig.add_trace(
            go.Bar(
                x=centers,
                y=cl_counts,
                width=binw * 0.95,
                marker=dict(color=COL_CLUSTER),
                opacity=ALPHA_CLUSTER,
                customdata=[{"kind": "hist_bin", "ids": b} for b in cl_bins],
                hovertext=[f"cluster bin ids: {b[:12]}{'...' if len(b) > 12 else ''}" for b in cl_bins],
                hoverinfo="text",
                showlegend=False,
            ),
            row=row,
            col=col
        )

        fig.add_trace(
            go.Bar(
                x=centers,
                y=fr_counts,
                width=binw * 0.95,
                marker=dict(color=COL_FREE),
                opacity=ALPHA_FREE,
                customdata=[{"kind": "hist_bin", "ids": b} for b in fr_bins],
                hovertext=[f"free bin ids: {b[:12]}{'...' if len(b) > 12 else ''}" for b in fr_bins],
                hoverinfo="text",
                showlegend=False,
            ),
            row=row,
            col=col
        )

        fig.update_xaxes(title_text=xlabel, row=row, col=col)
        fig.update_yaxes(title_text="count", row=row, col=col)


        if fit_pack.get("ok", False) and var_key in ("vx", "vy"):
            xgrid = np.linspace(float(edges[0]), float(edges[-1]), 1000)

            yfit = _gmm1d_from_2d_counts(
                x=xgrid,
                weights=np.asarray(fit_pack["weights"], float),
                means=np.asarray(fit_pack["means"], float),
                covs=np.asarray(fit_pack["covs"], float),
                n_total=int(fit_pack.get("n_used", 0)),
                binw=binw,
                axis=var_key,
            )

            fig.add_trace(
                go.Scatter(
                    x=xgrid,
                    y=yfit,
                    mode="lines",
                    line=dict(color="black", width=3),
                    hoverinfo="skip",
                    showlegend=False,
                ),
                row=row,
                col=col
            )

    add_hist("vx", row=1, col=4, xlabel="v_x (cm/s)")
    add_hist("speed", row=2, col=3, xlabel="speed (cm/s)")
    add_hist("vy", row=2, col=4, xlabel="v_y (cm/s)")
    add_current_temp_vx12_subplot(
        fig,
        vkey=vkey,
        t_min=t_min,
        min_points=min_points,
        cluster_n_threshold=cluster_n_threshold,
        free_s_threshold=free_s_threshold,
        include_clusters_in_fit=include_clusters_in_fit,
        row=3,
        col=3,
    )
    add_hist("n_est", row=3, col=4, xlabel="estimated particle number N")
    fig.update_xaxes(
        range=[None, 3.0],
        dtick=1,
        row=3,
        col=4,
        title_text="estimated particle number N",
    )
    fig.update_xaxes(range=[0.0, 1.0], row=1, col=1, title_text="s (straightness)")
    fig.update_yaxes(title_text="w = RMSD/(vΔt)", row=1, col=1)

    fig.update_xaxes(title_text="x (cm)", row=1, col=2)
    fig.update_yaxes(title_text="y (cm)", row=1, col=2)
    fig.update_xaxes(constrain="domain", row=1, col=2)
    fig.update_yaxes(scaleanchor="x2", scaleratio=1, constrain="domain", row=1, col=2)

    fig.update_xaxes(title_text="v_x (cm/s)", row=1, col=3)
    fig.update_yaxes(title_text="v_y (cm/s)", row=1, col=3)

    fig.update_xaxes(title_text="t (s)", row=2, col=1)
    fig.update_yaxes(title_text="x(t) (cm)", row=2, col=1)

    fig.update_xaxes(title_text="t (s)", row=3, col=1)
    fig.update_yaxes(title_text="y(t) (cm)", row=3, col=1)

    if selected_tid is not None and int(selected_tid) in vd.track_cache and int(selected_tid) in vis:
        ds = effective_track(vkey, int(selected_tid), t_min, cluster_n_threshold)
        fig.add_trace(go.Scatter(
            x=ds["x_plot"], y=ds["y_plot"], mode="lines",
            line=dict(color="black", width=5),
            hoverinfo="skip", showlegend=False,
        ), row=1, col=2)

        fig.add_trace(go.Scatter(
            x=ds["t_plot"], y=ds["x_plot"], mode="lines",
            line=dict(color="black", width=5),
            hoverinfo="skip", showlegend=False,
        ), row=2, col=1)

        fig.add_trace(go.Scatter(
            x=ds["t_plot"], y=ds["y_plot"], mode="lines",
            line=dict(color="black", width=5),
            hoverinfo="skip", showlegend=False,
        ), row=3, col=1)

        if np.isfinite(ds.get("vx", np.nan)) and np.isfinite(ds.get("vy", np.nan)):
            fig.add_trace(go.Scatter(
                x=[ds["vx"]], y=[ds["vy"]], mode="markers",
                marker=dict(size=18, color="rgba(0,0,0,0)", line=dict(color="black", width=2)),
                hoverinfo="skip", showlegend=False,
            ), row=1, col=3)

        if np.isfinite(ds.get("s_full", np.nan)) and np.isfinite(ds.get("w", np.nan)):
            fig.add_trace(go.Scatter(
                x=[ds["s_full"]], y=[ds["w"]],
                mode="markers",
                marker=dict(size=16, color="rgba(0,0,0,0)", line=dict(color="black", width=2)),
                hoverinfo="skip", showlegend=False,
            ), row=1, col=1)

    fig.update_layout(
        height=1380,
        margin=dict(l=55, r=25, t=90, b=140),
        hovermode="closest",
        barmode="stack",
        bargap=0.05,
        showlegend=True,
        legend=dict(
            x=0.600,
            y=0.200,
            xanchor="left",
            yanchor="top",
            orientation="v",
            bgcolor="rgba(255,255,255,0.78)",
            bordercolor="rgba(0,0,0,0.18)",
            borderwidth=1,
            font=dict(size=8),
            itemsizing="constant",
            itemwidth=30,
            tracegroupgap=2,
            traceorder="normal",
        ),
    )

    fit_text_lines = []
    fit_text_lines.append(f"video: {vkey}")
    fit_text_lines.append(f"min track length after t_min: {int(min_points)} points")
    if include_clusters_in_fit:
        fit_sample_rule_text = (
            "fit sample rule: SINGLE + CLUSTER trajectories can be used, "
            "provided they pass post-t_min, min-length, descending-stage, "
            "and straightness cuts"
        )
    else:
        fit_sample_rule_text = (
            "fit sample rule: SINGLE particles only + must have post-t_min data "
            "+ pass min-length cut + already be in descending stage"
        )

    fit_text_lines.append(fit_sample_rule_text)
    if fit_pack.get("ok", False):
        fit_text_lines.append(f"fit sample: n={fit_pack.get('n_used', 0)} tracks")
        fit_text_lines.append(
            f"include clusters in GMM fit: {include_clusters_in_fit}"
        )
        fit_text_lines.append(
            f"Size rule: cluster if N_est = (Lum/Lum_1)^(1/m) >= {float(cluster_n_threshold):g}, with m = {SIZE_M:g}."
        )
        fit_text_lines.append(
            f"Free-particle straightness rule: require s >= {float(free_s_threshold):g}."
        )
        fit_text_lines.append("Scatter fit model: 2D Gaussian mixture with 2 components in (vx, vy).")
        fit_text_lines.append("Histogram black curves for vx and vy are the corresponding 1D marginals from that fitted 2D GMM.")
        means = np.asarray(fit_pack["means"], float)
        covs = np.asarray(fit_pack["covs"], float)
        weights = np.asarray(fit_pack["weights"], float)

        for k in range(2):
            sigx = np.sqrt(max(float(covs[k, 0, 0]), 0.0))
            sigy = np.sqrt(max(float(covs[k, 1, 1]), 0.0))
            fit_text_lines.append(
                f"GMM comp {k}: weight={weights[k]:.4g}, "
                f"mu_x={means[k,0]:.4g}, sigma_x={sigx:.4g}, "
                f"mu_y={means[k,1]:.4g}, sigma_y={sigy:.4g}"
            )
        cf = fit_pack.get("center_free", [np.nan, np.nan])
        ct = fit_pack.get("center_trapped", [np.nan, np.nan])
        fit_text_lines.append(
            f"2D centers: free(left-bottom)=({cf[0]:.4g}, {cf[1]:.4g}), "
            f"trapped(right-top)=({ct[0]:.4g}, {ct[1]:.4g})"
        )
        tau_pack = relaxation_time_from_free_gmm(fit_pack)

        if tau_pack.get("ok", False):
            fit_text_lines.append("")
            fit_text_lines.append("Relaxation time from left-bottom/free GMM component:")
            fit_text_lines.append(
                f"mu_y_free = {tau_pack['mu_y_cm_s']:.4g} cm/s, "
                f"sigma_y_free = {tau_pack['sigma_y_cm_s']:.4g} cm/s"
            )
            fit_text_lines.append(
                f"tau_app = |mu_y_free| / g' = "
                f"{tau_pack['tau_ms']:.4g} ± {tau_pack['tau_err_ms']:.4g} ms"
            )
            fit_text_lines.append(
                f"using g' = {G_PRIME_CM_S2:.4g} cm/s^2"
            )
            fit_text_lines.append(
                f"optional SEM of fitted centre: ± {tau_pack['tau_sem_ms']:.4g} ms "
                f"(n_eff ≈ {tau_pack['n_eff']:.4g})"
            )
        else:
            fit_text_lines.append("")
            fit_text_lines.append(
                f"Relaxation time calculation failed: {tau_pack.get('message', '')}"
            )
        fit_text_lines.append(
    "Classification rule: cluster-labelled trajectories are displayed separately; "
    "non-cluster trajectories are classified by the fitted 2D GMM in (vx, vy)."
)
        fit_text_lines.append("Heatmap overlay: still displayed from fitted 2D diagonal peaks for visualization.")
    else:
        fit_text_lines.append("fit: FAILED")
        fit_text_lines.append(str(fit_pack.get("message", "")))

    return fig, vis, "\n".join(fit_text_lines)


# ============================================================
# --------------------------- UI ------------------------------
# ============================================================

app = Dash(__name__)
app.title = "Tracks Viewer (short) — only manually_added__xfm + ALL_COMBINED"

motion_options = [{"label": "free", "value": "free"}, {"label": "trapped", "value": "trapped"}]
size_options = [{"label": "single", "value": "single"}, {"label": "cluster", "value": "cluster"}]

app.layout = html.Div(
    style={"fontFamily": "Arial, sans-serif", "padding": "10px"},
    children=[
        html.Div(
            style={"display": "flex", "gap": "20px", "alignItems": "flex-start", "flexWrap": "wrap"},
            children=[
                html.Div(
                    style={"minWidth": "520px"},
                    children=[
                        html.Div("Video:", style={"fontWeight": "bold"}),
                        dcc.Dropdown(
                            id="dd-video",
                            options=[{"label": k, "value": k} for k in VIDEO_KEYS],
                            value=REFERENCE_VIDEO_KEY,
                            clearable=False,
                            searchable=True,
                        ),
                        html.Div(style={"height": "8px"}),
                        html.Div("Transient removal (t-selector):", style={"fontWeight": "bold"}),
                        html.Div(
                            style={"display": "flex", "gap": "10px", "alignItems": "center"},
                            children=[
                                html.Div("Use only data with t ≥", style={"fontWeight": "bold"}),
                                dcc.Input(id="in-t-min", type="number", value=T_MIN_DEFAULT, debounce=True, style={"width": "90px"}),
                                html.Div("seconds", style={"color": "#444"}),
                            ],
                        ),
                        html.Div(style={"height": "8px"}),
                        html.Div("Track length filter:", style={"fontWeight": "bold"}),
                        html.Div(style={"height": "8px"}),
                        html.Div("Cluster / free thresholds:", style={"fontWeight": "bold"}),
                        html.Div(
                            style={"display": "flex", "gap": "18px", "alignItems": "center", "flexWrap": "wrap"},
                            children=[
                                html.Div(
                                    style={"display": "flex", "gap": "8px", "alignItems": "center"},
                                    children=[
                                        html.Div("Cluster if N ≥", style={"fontWeight": "bold"}),
                                        dcc.Input(
                                            id="in-cluster-n-threshold",
                                            type="number",
                                            value=CLUSTER_N_THRESHOLD_DEFAULT,
                                            debounce=True,
                                            step=0.1,
                                            style={"width": "90px"}
                                        ),
                                    ],
                                ),
                                html.Div(
                                    style={"display": "flex", "gap": "8px", "alignItems": "center"},
                                    children=[
                                        html.Div("Free if s ≥", style={"fontWeight": "bold"}),
                                        dcc.Input(
                                            id="in-free-s-threshold",
                                            type="number",
                                            value=FREE_S_THRESHOLD_DEFAULT,
                                            debounce=True,
                                            step=0.01,
                                            style={"width": "90px"}
                                        ),
                                    ],
                                ),
                            ],
                        ),
                        html.Div(
                            style={"display": "flex", "gap": "10px", "alignItems": "center"},
                            children=[
                                html.Div("Use only tracks with at least", style={"fontWeight": "bold"}),
                                dcc.Input(id="in-min-points", type="number", value=MIN_POINTS_DEFAULT, debounce=True, style={"width": "90px"}),
                                html.Div("points after t ≥ t_min", style={"color": "#444"}),
                            ],
                        ),
                        html.Div(style={"height": "8px"}),
                        dcc.Checklist(
                            id="ck-show-heatmap",
                            options=[{"label": "Overlay fitted heatmap on Vy vs Vx", "value": "on"}],
                            value=["on"],
                            inline=True
                        ),
                        html.Div(style={"height": "8px"}),
                        dcc.Checklist(
                            id="ck-include-clusters-fit",
                            options=[
                                {
                                    "label": "Include clusters in GMM fitting",
                                    "value": "on",
                                }
                            ],
                            value=(["on"] if INCLUDE_CLUSTERS_IN_GMM_DEFAULT else []),
                            inline=True,
                        ),
                        html.Div(
                            id="fit-text",
                            style={
                                "whiteSpace": "pre-line",
                                "fontFamily": "Consolas, monospace",
                                "marginTop": "8px",
                            },
                        ),
                        html.Div(
                            id="info",
                            style={
                                "whiteSpace": "pre-line",
                                "fontFamily": "Consolas, monospace",
                                "marginTop": "8px",
                            },
                        ),
                    ],
                ),
                html.Div([
                    html.Div("Show motion:", style={"fontWeight": "bold"}),
                    dcc.Checklist(id="ck-show-motion", options=motion_options, value=["free", "trapped"], inline=True),
                ]),
                html.Div([
                    html.Div("Show size:", style={"fontWeight": "bold"}),
                    dcc.Checklist(id="ck-show-size", options=size_options, value=["single", "cluster"], inline=True),
                ]),
                html.Button("Clear highlight", id="btn-clear", n_clicks=0),
                html.Button(
                    "Delete selected track",
                    id="btn-delete-track",
                    n_clicks=0,
                    style={"marginLeft": "8px", "backgroundColor": "#c62828", "color": "white"},
                ),
                html.Div(
                    id="delete-msg",
                    style={
                        "whiteSpace": "pre-line",
                        "minWidth": "520px",
                        "color": "#b00020",
                    },
                ),
            ],
        ),
        html.Hr(),
        dcc.Store(id="store-selected", data=None),
        dcc.Graph(id="graph", figure=go.Figure(), style={"height": "1300px"},
                  config={"scrollZoom": True, "displayModeBar": True, "responsive": True}),
        html.Hr(),
        html.Div(
            style={
                "display": "flex",
                "flexDirection": "column",
                "gap": "20px",
                "alignItems": "center",
            },
            children=[
                dcc.Graph(
                    id="graph-vx1-laser-alltemps",
                    figure=go.Figure(),
                    style={"height": "520px", "width": "520px"},
                    config={"scrollZoom": True, "displayModeBar": True, "responsive": True},
                ),
                dcc.Graph(
                    id="graph-vx2-laser-alltemps",
                    figure=go.Figure(),
                    style={"height": "520px", "width": "520px"},
                    config={"scrollZoom": True, "displayModeBar": True, "responsive": True},
                ),
            ],
        ),
    ],
)


@app.callback(
    Output("store-selected", "data"),
    Input("graph", "clickData"),
    Input("btn-clear", "n_clicks"),
    State("store-selected", "data"),
    prevent_initial_call=True
)
def selection_controller(clickData, n_clear, cur):
    trig = getattr(dash.ctx, "triggered_id", None)
    if trig == "btn-clear":
        return None

    if trig == "graph":
        if not clickData or "points" not in clickData or not clickData["points"]:
            return no_update
        pt = clickData["points"][0]
        cd = pt.get("customdata", None)

        if isinstance(cd, dict) and cd.get("kind") == "hist_bin":
            ids = cd.get("ids", []) or []
            ids = [int(x) for x in ids if isinstance(x, (int, float, np.integer)) and np.isfinite(x)]
            if len(ids) == 1:
                return ids[0]
            return no_update

        if isinstance(cd, (int, np.integer)):
            tid = int(cd)
            return None if (cur == tid) else tid
        if isinstance(cd, float) and np.isfinite(cd):
            tid = int(cd)
            return None if (cur == tid) else tid

    return no_update


@app.callback(
    Output("delete-msg", "children"),
    Output("dd-video", "value", allow_duplicate=True),
    Output("store-selected", "data", allow_duplicate=True),
    Input("btn-delete-track", "n_clicks"),
    State("dd-video", "value"),
    State("store-selected", "data"),
    prevent_initial_call=True
)
def delete_selected_track(n_clicks, vkey, selected_tid):
    if not n_clicks:
        return no_update, no_update, no_update

    if vkey is None:
        return "No video selected.", no_update, no_update

    if selected_tid is None:
        return "No selected track to delete.", no_update, no_update

    if vkey == ALL_KEY:
        return "Delete is disabled for ALL_COMBINED. Please delete in the specific per-video file.", no_update, no_update

    ok, msg = delete_track_and_save_csv(vkey, int(selected_tid))
    if not ok:
        return msg, no_update, no_update

    vd_old = VIDEO_DATA[vkey]
    try:
        df_tr = pd.read_csv(vd_old.deleted_tracks_csv)
        track_cache, t_end_max, lum1 = compute_track_level_metrics_from_tracks(
            df_tr, CM_PER_PIXEL, FPS
        )

        VIDEO_DATA[vkey] = VideoData(
            key=vkey,
            tracks_csv=vd_old.deleted_tracks_csv,
            original_tracks_csv=vd_old.original_tracks_csv,
            deleted_tracks_csv=vd_old.deleted_tracks_csv,
            tp_csv=vd_old.tp_csv,
            track_cache=track_cache,
            t_end_max=t_end_max,
            lum1=lum1,
        )
        track_cache_all, t_end_max_all = build_all_combined_track_cache(VIDEO_DATA)
        VIDEO_DATA[ALL_KEY] = VideoData(
            key=ALL_KEY,
            tracks_csv="(combined) " + BASE_DIR,
            original_tracks_csv="(combined)",
            deleted_tracks_csv="(combined)",
            tp_csv="(not used)",
            track_cache=track_cache_all,
            t_end_max=t_end_max_all,
            lum1=np.nan,
        )

    except Exception as e:
        return f"{msg}\nBut reload failed: {e}", no_update, None

    return msg, vkey, None

@app.callback(
    Output("graph-vx1-laser-alltemps", "figure"),
    Output("graph-vx2-laser-alltemps", "figure"),
    Input("in-t-min", "value"),
    Input("in-min-points", "value"),
    Input("in-cluster-n-threshold", "value"),
    Input("in-free-s-threshold", "value"),
    Input("ck-include-clusters-fit", "value"),
)
def redraw_alltemps_figures(
    t_min,
    min_points,
    cluster_n_threshold,
    free_s_threshold,
    ck_include_clusters_fit,
):
    include_clusters_in_fit = "on" in (ck_include_clusters_fit or [])
    free_s_threshold = (
        float(free_s_threshold)
        if (free_s_threshold is not None and np.isfinite(free_s_threshold))
        else FREE_S_THRESHOLD_DEFAULT
    )
    
    t_min = float(t_min) if (t_min is not None and np.isfinite(t_min)) else T_MIN_DEFAULT
    min_points = int(min_points) if (min_points is not None and np.isfinite(min_points)) else MIN_POINTS_DEFAULT
    cluster_n_threshold = (
        float(cluster_n_threshold)
        if (cluster_n_threshold is not None and np.isfinite(cluster_n_threshold))
        else CLUSTER_N_THRESHOLD_DEFAULT
    )

    fig1 = build_vx1_vs_laser_figure_alltemps(
        t_min,
        min_points,
        cluster_n_threshold,
        free_s_threshold,
        include_clusters_in_fit=include_clusters_in_fit,
    )
    fig2 = build_vx2_vs_laser_figure_alltemps(
        t_min,
        min_points,
        cluster_n_threshold,
        free_s_threshold,
        include_clusters_in_fit=include_clusters_in_fit,
    )
    return fig1, fig2
    
@app.callback(
    Output("graph", "figure"),
    Output("fit-text", "children"),
    Output("info", "children"),
    Input("dd-video", "value"),
    Input("store-selected", "data"),
    Input("ck-show-motion", "value"),
    Input("ck-show-size", "value"),
    Input("in-t-min", "value"),
    Input("in-min-points", "value"),
    Input("in-cluster-n-threshold", "value"),
    Input("in-free-s-threshold", "value"),
    Input("ck-include-clusters-fit", "value"),
    Input("ck-show-heatmap", "value"),
)
def redraw(
    vkey,
    selected_tid,
    motion_vals,
    size_vals,
    t_min,
    min_points,
    cluster_n_threshold,
    free_s_threshold,
    ck_include_clusters_fit,
    ck_heatmap,
):
    cluster_n_threshold = (
        float(cluster_n_threshold)
        if (cluster_n_threshold is not None and np.isfinite(cluster_n_threshold))
        else CLUSTER_N_THRESHOLD_DEFAULT
    )
    free_s_threshold = (
        float(free_s_threshold)
        if (free_s_threshold is not None and np.isfinite(free_s_threshold))
        else FREE_S_THRESHOLD_DEFAULT
    )
    if vkey is None:
        return go.Figure(), "", "No video selected."

    t_min = float(t_min) if (t_min is not None and np.isfinite(t_min)) else T_MIN_DEFAULT
    min_points = int(min_points) if (min_points is not None and np.isfinite(min_points)) else MIN_POINTS_DEFAULT    
    show_motion = set(motion_vals or [])
    show_size = set(size_vals or [])
    include_clusters_in_fit = "on" in (ck_include_clusters_fit or [])
    show_heatmap = ("on" in (ck_heatmap or []))

    fig, vis, fit_text = build_figure(
        vkey=vkey,
        selected_tid=(int(selected_tid) if selected_tid is not None else None),
        show_motion=show_motion,
        show_size=show_size,
        t_min=t_min,
        min_points=min_points,
        cluster_n_threshold=cluster_n_threshold,
        free_s_threshold=free_s_threshold,
        include_clusters_in_fit=include_clusters_in_fit,
        show_heatmap=show_heatmap,
    )


    vd = VIDEO_DATA[vkey]

    selected_extra = ""
    if selected_tid is not None and int(selected_tid) in vd.track_cache:
        ds = effective_track(vkey, int(selected_tid), t_min, cluster_n_threshold)
        fit_pack_dbg = compute_fit_pack(
            vkey,
            t_min,
            min_points,
            cluster_n_threshold,
            free_s_threshold,
            include_clusters_in_fit=include_clusters_in_fit,
        )
        final_cls_dbg, reason_dbg = explain_selected_class(
            ds, fit_pack_dbg, min_points, free_s_threshold
        )

        try:
            y_plot_dbg = np.asarray(ds.get("y_plot", []), float)
            if y_plot_dbg.size > 0:
                i_peak_dbg = int(np.nanargmax(y_plot_dbg))
                tail_pts_dbg = int(len(y_plot_dbg) - (i_peak_dbg + 1))
            else:
                i_peak_dbg = -1
                tail_pts_dbg = 0
        except Exception:
            i_peak_dbg = -1
            tail_pts_dbg = 0

        selected_extra = (
            f"Selected lum         : {ds.get('lum', np.nan):.4g}\n"
            f"Selected Lum1        : {ds.get('lum1', np.nan):.4g}\n"
            f"Selected N           : {ds.get('n_est', np.nan):.4g}\n"
            f"Selected R           : {ds.get('r_est_um', np.nan):.4g} um\n"
            f"Selected size class  : {'cluster' if ds.get('is_cluster', False) else 'single'}\n"
            f"Selected final class : {final_cls_dbg}\n"
            f"Selected reason      : {reason_dbg}\n"
            f"Selected vx          : {ds.get('vx', np.nan):.4g}\n"
            f"Selected vy          : {ds.get('vy', np.nan):.4g}\n"
            f"Descending stage     : {ds.get('reached_descending_stage', False)}\n"
            f"Points post t_min    : {ds.get('n_points_post_tmin', 0)}\n"
            f"Tail pts after ypeak : {tail_pts_dbg}\n"
        )

    lum1_text = (
        f"Lum_1     : {vd.lum1:.4g}\n"
        if np.isfinite(vd.lum1)
        else "Lum_1     : (per-video normalisation in ALL_COMBINED)\n"
    )

    info = (
        f"t_min     : {t_min:g} s\n"
        f"min_pts   : {min_points}\n"
        f"cluster N : {cluster_n_threshold:g}\n"
        f"free s    : {free_s_threshold:g}\n"
        f"clusters in GMM fit : {include_clusters_in_fit}\n"
        f"{lum1_text}"
        f"Visible   : {len(vis)} tracks "
        f"(motion={sorted(show_motion)}, size={sorted(show_size)})\n"
        f"Selected  : {selected_tid if selected_tid is not None else '(none)'}\n"
        f"{selected_extra}"
    )

    return fig, fit_text, info


# ============================================================
# --------------------------- main ----------------------------
# ============================================================

if __name__ == "__main__":
    print("Dash app starting...")
    print(f"BASE_DIR = {BASE_DIR}")
    print(f"Only files ending with: {ONLY_SUFFIX}")
    print(f"Found per-param videos: {len(VIDEO_CONFIGS)}")
    print(f"Dropdown includes: {len(VIDEO_KEYS)} options (incl. {ALL_KEY})")
    print(f"Local: http://{HOST}:{PORT}")
    app.run(host=HOST, port=PORT, debug=DEBUG)

Dash app starting...
BASE_DIR = D:\Manchester\Manchester\year four\lab\Input Videos\videos 2
Only files ending with: __manually_added__linkfmt__subpixel__xfm.csv
Found per-param videos: 17
Dropdown includes: 18 options (incl. ALL_COMBINED (all params))
Local: http://127.0.0.1:8060
